# Web scrapping for books


#### Step 1: Import Libraries and Fetch the Page

In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

#### Step 2: Send an HTTP Request and Check the Response

In [14]:
url = 'https://books.toscrape.com/'
response = requests.get(url)

# Check if request was successful (200 means success)
print("Status Code:", response.status_code)

Status Code: 200


#### Step 3: Parse the HTML with BeautifulSoup
Convert the raw HTML text into a structured BeautifulSoup object so we can navigate tags.

In [15]:
soup = BeautifulSoup(response.text, 'html.parser')

# Preview the title of the webpage
print("Page Title:", soup.title.text.strip())

Page Title: All products | Books to Scrape - Sandbox


#### Step 4: Inspect and Extract Data for a Single Book
On the website, every book is enclosed in an <article> tag with the class product_pod. Let's isolate the first book to test our extraction logic.

In [16]:
# From HTML: <article class="product_pod">
book = soup.find('article', class_='product_pod')

# From HTML: <a href="..." title="A Light in the Attic">A Light in the Attic</a>
title = book.h3.a['title']

# From HTML: <p class="price_color">£51.77</p>
price = book.find('p', class_='price_color').text

# From HTML: <p class="star-rating Three"></p>
rating = book.find('p', class_='star-rating')['class'][1]    # extracts "Three" after finding the class

print(f"Title: {title} | Price: {price} | Rating: {rating}")

Title: A Light in the Attic | Price: Â£51.77 | Rating: Three


#### Step 5: Loop Through All Books on the Page

In [17]:
books_data = []

# From HTML: <article class="product_pod"> (Repeated for every book listed)
books = soup.find_all('article', class_='product_pod')

for b in books:
    # From HTML: <a title="Book Title">
    title = b.h3.a['title']
    
    # From HTML: <p class="price_color">£XX.XX</p>
    price = b.find('p', class_='price_color').text
    
    # From HTML: <p class="star-rating RatingName">
    rating = b.find('p', class_='star-rating')['class'][1]
    
    books_data.append({
        'Title': title, 
        'Price': price, 
        'Rating': rating
    })

#### Step 6: Convert and Display in Pandas DataFrame

In [18]:
df = pd.DataFrame(books_data)

# Display the first 5 rows in Jupyter Lab
df

,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five
5,The Requiem Red,Â£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four
9,The Black Maria,Â£52.15,One


In [21]:
# Use ExcelWriter with mode='a' to open the existing file
# if_sheet_exists='replace' will update/overwrite just the "Books" sheet inside the existing workbook
excel_filename = "Scrapped.xlsx"
with pd.ExcelWriter(excel_filename, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, index=False, sheet_name="Books")

print(f"Data successfully updated inside existing file {excel_filename}!")

Data successfully updated inside existing file Scrapped.xlsx!
